# Convert CSV Embeddings to NPZ

Convert paired embedding mean and variance CSV files into an `.npz` file that can be used by `config.embeddings.standard.embedding_init_file`.

The saved archive includes three keys:
- `embeddings`: mean embeddings, matching the current standard initializer.
- `mu`: the same mean embeddings, matching saved embedding snapshots.
- `var`: variance embeddings from the variance CSV.


## Configure Paths

Change these paths for a different experiment output. The default paths point to the `hq_train100` CSV files in `saved_results/`.

In [6]:
from __future__ import annotations

from pathlib import Path

import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "experiments").exists() and (candidate / "saved_results").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()

mean_csv = REPO_ROOT / "saved_results" / "hq_embeddings_mean.csv"
var_csv = REPO_ROOT / "saved_results" / "hq_embeddings_var.csv"
out_npz = REPO_ROOT / "saved_results" / "hq_embeddings.npz"

strict_finite = False

mean_csv, var_csv, out_npz


(PosixPath('/home/dw/cuTAGI_DW/saved_results/hq_embeddings_mean.csv'),
 PosixPath('/home/dw/cuTAGI_DW/saved_results/hq_embeddings_var.csv'),
 PosixPath('/home/dw/cuTAGI_DW/saved_results/hq_embeddings.npz'))

## Load CSV Files

The loader accepts plain numeric CSVs and CSVs that may contain all-empty rows or columns. Values are saved as `float32`, which matches `EmbeddingLayer`.

In [7]:
def load_embedding_csv(path: Path) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(path)

    array = np.genfromtxt(path, delimiter=",", dtype=np.float32)
    array = np.atleast_2d(array)

    # Remove entirely empty rows/columns when at least one data row/column remains.
    if array.size:
        row_keep = ~np.all(np.isnan(array), axis=1)
        col_keep = ~np.all(np.isnan(array), axis=0)
        if row_keep.any():
            array = array[row_keep]
        if col_keep.any():
            array = array[:, col_keep]

    if array.ndim != 2 or array.size == 0:
        raise ValueError(f"{path} did not load as a non-empty 2D matrix")

    return array.astype(np.float32, copy=False)


mean = load_embedding_csv(mean_csv)
var = load_embedding_csv(var_csv)

{
    "mean_shape": mean.shape,
    "var_shape": var.shape,
    "mean_dtype": str(mean.dtype),
    "var_dtype": str(var.dtype),
}


{'mean_shape': (100, 20),
 'var_shape': (100, 20),
 'mean_dtype': 'float32',
 'var_dtype': 'float32'}

## Validate Shapes and Values

The mean and variance matrices must have exactly the same shape. Non-finite values are reported; set `strict_finite = True` in the configuration cell if the conversion should stop when `NaN` or `inf` values are present.

In [8]:
if mean.shape != var.shape:
    raise ValueError(f"Shape mismatch: mean {mean.shape} vs var {var.shape}")

mean_finite = np.isfinite(mean)
var_finite = np.isfinite(var)

summary = {
    "embedding_shape": mean.shape,
    "mean_nonfinite_count": int(mean.size - mean_finite.sum()),
    "var_nonfinite_count": int(var.size - var_finite.sum()),
    "mean_min": float(np.nanmin(mean)) if mean_finite.any() else None,
    "mean_max": float(np.nanmax(mean)) if mean_finite.any() else None,
    "var_min": float(np.nanmin(var)) if var_finite.any() else None,
    "var_max": float(np.nanmax(var)) if var_finite.any() else None,
}

if strict_finite and (
    summary["mean_nonfinite_count"] or summary["var_nonfinite_count"]
):
    raise ValueError(f"Non-finite values found: {summary}")

summary


{'embedding_shape': (100, 20),
 'mean_nonfinite_count': 0,
 'var_nonfinite_count': 0,
 'mean_min': -0.22635363042354584,
 'mean_max': 0.161751851439476,
 'var_min': 0.0016243973514065146,
 'var_max': 0.007601599674671888}

## Save NPZ

Use `out_npz` as `config.embeddings.standard.embedding_init_file`. The current initializer reads the `embeddings` key; `mu` and `var` are also stored for compatibility with embedding snapshots.

In [9]:
out_npz.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    out_npz,
    embeddings=mean,
    mu=mean,
    var=var,
)

out_npz


PosixPath('/home/dw/cuTAGI_DW/saved_results/hq_embeddings.npz')

## Reload Check

This confirms the archive contains the expected keys and shapes.

In [10]:
with np.load(out_npz) as data:
    check = {
        "path": str(out_npz),
        "keys": sorted(data.files),
        "embeddings_shape": data["embeddings"].shape,
        "mu_shape": data["mu"].shape,
        "var_shape": data["var"].shape,
        "embeddings_dtype": str(data["embeddings"].dtype),
        "var_dtype": str(data["var"].dtype),
    }

check


{'path': '/home/dw/cuTAGI_DW/saved_results/hq_embeddings.npz',
 'keys': ['embeddings', 'mu', 'var'],
 'embeddings_shape': (100, 20),
 'mu_shape': (100, 20),
 'var_shape': (100, 20),
 'embeddings_dtype': 'float32',
 'var_dtype': 'float32'}

Config example:

```yaml
embeddings:
  standard:
    embedding_init_file: saved_results/hq_train100_embeddings.npz
```